# 03 - LLM KV Cache 与流式输入实验

本 notebook 用于演示和实验大型语言模型 (LLM) 的 KV 缓存机制，以及如何结合流式输入（模拟 ASR 输出）来加速首个 token 的生成。
主要内容包括：
1. 初始化 `StreamLLMInference` 类，加载 LLM 和分词器。
2. 演示为初始提示 (prompt) 预计算 KV 缓存。
3. 模拟流式输入文本片段，并逐步更新 KV 缓存，然后从更新后的缓存生成下一个 token。
4. 比较使用 KV 缓存与不使用 KV 缓存（即每次都完整处理输入）的 token 生成延迟。
5. 记录和分析 KV 缓存大小及相关延迟指标。

## 1. 初始化与配置

In [ ]:
import os
import sys
import time
import numpy as np
import torch # 确保 torch 可用

# 将项目根目录添加到 sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.config import LLM_MODEL_NAME, DEVICE
from src.llm.stream_llm_inference import StreamLLMInference
from src.utils.logging_utils import get_logger

logger = get_logger('LLMKVCacheNotebook')

# 检查GPU是否可用 (如果DEVICE配置为cuda)
if "cuda" in DEVICE and not torch.cuda.is_available():
    logger.warning(f"PyTorch CUDA 未检测到，但DEVICE设置为 {DEVICE}. 将尝试使用CPU。")
    effective_device = "cpu"
else:
    effective_device = DEVICE
logger.info(f"LLM 模型: {LLM_MODEL_NAME}")
logger.info(f"运行设备: {effective_device}")

## 2. 初始化 StreamLLMInference

In [ ]:
try:
    llm_streamer = StreamLLMInference(
        model_name_or_path=LLM_MODEL_NAME,
        device=effective_device,
        # torch_dtype_str="float16" # 如果显存充足且GPU支持，可以考虑使用半精度
    )
    logger.info(f"StreamLLMInference 初始化成功。模型: {llm_streamer.model_name_or_path}, 设备: {llm_streamer.device}")
except Exception as e:
    logger.error(f"初始化 StreamLLMInference 失败: {e}", exc_info=True)
    llm_streamer = None

## 3. 实验场景定义

我们将模拟一个逐步接收 ASR 输出的场景。
- **初始提示 (Initial Prompt)**: 可能是一个系统指令或者对话的开始。
- **文本片段 (Text Fragments)**: 模拟 ASR 识别出的后续文本块。

In [ ]:
initial_prompt_text = "用户问：你好，今天天气怎么样？\n助手答："
# 模拟ASR后续识别到的文本片段 (逐步输入)
text_fragments_stream = [
    "今天",
    "天气很",
    "好，阳光",
    "明媚，",
    "适合出去玩。"
]

full_input_text_for_comparison = initial_prompt_text + "".join(text_fragments_stream)

logger.info(f"初始提示: '{initial_prompt_text}'")
logger.info(f"文本片段流: {text_fragments_stream}")
logger.info(f"完整输入 (对比用): '{full_input_text_for_comparison}'")

## 4. 实验 1: 使用 KV 缓存进行流式推理

步骤：
1. 对 `initial_prompt_text` 预计算 KV 缓存。
2. 逐个接收 `text_fragments_stream` 中的片段：
   a. 使用当前片段更新 KV 缓存。
   b. 从更新后的 KV 缓存生成下一个 token（或少量 tokens）。
3. 记录首个 token 生成延迟、每次更新和生成的耗时、KV 缓存大小。

In [ ]:
if llm_streamer:
    logger.info("\n--- 实验 1: 使用 KV 缓存进行流式推理 ---")
    llm_streamer.reset_state() # 清空之前的状态和KV缓存
    
    # 1. 预计算初始提示的 KV 缓存
    logger.info(f"为初始提示预计算 KV 缓存: '{initial_prompt_text}'")
    start_precompute_time = time.time()
    precompute_input_ids = llm_streamer._prepare_inputs(initial_prompt_text)
    _, past_key_values = llm_streamer.precompute_kv_cache_for_prompt(initial_prompt_text)
    end_precompute_time = time.time()
    precompute_duration_ms = (end_precompute_time - start_precompute_time) * 1000
    logger.info(f"  初始提示预计算完成。耗时: {precompute_duration_ms:.2f} ms")
    llm_streamer._log_kv_cache_size(past_key_values, "初始提示预计算后")
    
    # 用于存储生成的 token 序列
    generated_tokens_with_cache = []
    # 用于存储每次生成操作的耗时
    token_gen_latencies_ms = []
    # 用于存储每次KV缓存更新的耗时 (如果 StreamLLMInference 中拆分了该计时)
    kv_update_latencies_ms = [] 
    
    current_input_text = initial_prompt_text
    first_token_generated = False
    first_token_latency_ms = -1
    overall_stream_start_time = time.time() # 从第一个片段开始计时FTL
    
    # 2. 模拟流式输入并生成
    for i, fragment in enumerate(text_fragments_stream):
        logger.info(f"  接收流式片段 {i+1}: '{fragment}'")
        current_input_text += fragment
        
        # a. 使用新片段更新KV缓存，并准备生成 (generate_next_token 内部会处理)
        # 注意: StreamLLMInference.generate_next_token 设计为接收的是 *新* 的文本片段
        # 它内部会与已有的 past_key_values 结合
        
        # b. 生成下一个 token (或少量 tokens, 这里演示生成一个)
        gen_start_time = time.time()
        # generate_next_token 返回 (next_token_str, new_past_key_values)
        # 这里我们假设每次流式输入后都想生成一个 token
        # 实际 pipeline 中可能需要更复杂的逻辑决定何时生成
        next_token, latency_info = llm_streamer.generate_next_token(new_text_fragment=fragment, temperature=0.1, top_p=0.9)
        gen_end_time = time.time()
        step_latency_ms = (gen_end_time - gen_start_time) * 1000
        
        if latency_info["last_token_gen_time_ms"] > 0:
             token_gen_latencies_ms.append(latency_info["last_token_gen_time_ms"])
        if latency_info["last_precompute_time_ms"] > 0: # 如果更新KV缓存的耗时被分开了
             kv_update_latencies_ms.append(latency_info["last_precompute_time_ms"])
        else: # 否则，我们认为 step_latency_ms 主要就是 token 生成（包含了可能的内部 cache 更新）
             if not latency_info["last_token_gen_time_ms"] > 0 and next_token: # 确保是生成token的耗时
                # 如果模型没分开计时，就用外部测量的这个步骤的总耗时 (减去可能的 tokenization 时间)
                # 为简化，这里用模型报告的，如果不可用则用粗略的外部计时
                token_gen_latencies_ms.append(step_latency_ms if step_latency_ms > 0 else 0.1) # 避免0
        
        if next_token:
            logger.info(f"    生成 Token: '{next_token}', 耗时(模型报告): {latency_info['last_token_gen_time_ms']:.2f}ms (含KV更新部分耗时: {latency_info['last_precompute_time_ms']:.2f}ms)")
            generated_tokens_with_cache.append(next_token)
            if not first_token_generated:
                first_token_latency_ms = (gen_end_time - overall_stream_start_time) * 1000
                logger.info(f"    !!! 首个 Token 生成 !!! 延迟: {first_token_latency_ms:.2f} ms (从首个片段输入开始计算)")
                first_token_generated = True
        else:
            logger.info("    此步骤未生成 token (可能输入只是更新了缓存)。")
        
        llm_streamer._log_kv_cache_size(llm_streamer.past_key_values, f"片段 {i+1} 处理后")
        
        # 可以在这里加一个小的暂停，模拟真实场景下ASR片段间的间隔
        # time.sleep(0.1)
        if i == 0 and not first_token_generated: # 更新整体流开始时间为第一个片段处理后
            overall_stream_start_time = time.time()

    final_response_with_cache = initial_prompt_text + "".join(generated_tokens_with_cache)
    logger.info(f"  最终流式生成结果 (带缓存): '{final_response_with_cache}'")
    if first_token_latency_ms > 0:
        logger.info(f"  首个 Token 延迟 (从首个片段输入开始): {first_token_latency_ms:.2f} ms")
    if token_gen_latencies_ms:
        logger.info(f"  平均每 Token 生成耗时 (模型报告): {np.mean(token_gen_latencies_ms):.2f} ms")
    if kv_update_latencies_ms:
        logger.info(f"  平均 KV 缓存更新耗时 (模型报告，如果分离): {np.mean(kv_update_latencies_ms):.2f} ms")
else:
    logger.error("LLM Streamer 未初始化，跳过实验 1。")

### 4.1 (补充) 演示 `generate_full_response_with_cache`

这个方法封装了对初始提示的KV缓存预计算，以及后续对一个或多个文本片段的流式处理和token生成。

In [ ]:
if llm_streamer:
    logger.info("\n--- 实验 1.1: 使用 generate_full_response_with_cache ---")
    llm_streamer.reset_state()
    
    # 将所有片段合并为一个，模拟一次性给出所有后续文本
    # 或者可以分多次调用 generate_full_response_with_cache (但不符合其设计意图)
    # 这里我们演示：先有 initial_prompt, 然后一次性来了所有 text_fragments_stream
    # StreamLLMInference.generate_full_response_with_cache 期望的是：
    #   - initial_prompt_text: 用于预填充KV缓存的文本
    #   - subsequent_text_fragments: 一个新文本片段的列表，LLM将基于这些片段和已缓存的KV继续生成

    logger.info(f"测试 generate_full_response_with_cache: initial_prompt='{initial_prompt_text}', subsequent_fragments={text_fragments_stream}")
    
    # 模拟：先对 initial_prompt_text 预计算，然后对每个 fragment 生成一个 token
    # 注意：generate_full_response_with_cache 的主要目的是对 initial_prompt_text 预计算，
    # 然后对 *第一个* subsequent_text_fragments 生成，或者如果 subsequent_text_fragments 为空，就直接从 initial_prompt_text 生成。
    # 为了模拟对多个片段的逐个响应，我们需要多次调用 generate_next_token，如实验1所示。
    # 这里，我们仅演示其基本用法：预计算+对第一个后续片段生成token
    
    llm_streamer.reset_state()
    start_full_response_time = time.time()
    
    # 步骤1: 预计算初始提示 (方法内部完成)
    # 步骤2: 对于第一个后续片段生成 (方法内部完成)
    # 我们期望它能对 initial_prompt_text 缓存，然后处理 text_fragments_stream[0] 并生成第一个 token
    generated_output, timings = llm_streamer.generate_full_response_with_cache(
        initial_prompt_text=initial_prompt_text,
        subsequent_text_fragments=[text_fragments_stream[0]], # 只给第一个片段
        max_new_tokens=1, # 只生成一个 token
        temperature=0.1, top_p=0.9
    )
    end_full_response_time = time.time()
    full_response_duration_ms = (end_full_response_time - start_full_response_time) * 1000

    logger.info(f"  generate_full_response_with_cache 输出: '{generated_output}'")
    logger.info(f"  总耗时: {full_response_duration_ms:.2f} ms")
    logger.info(f"  计时细节: precompute_kv_ms={timings['precompute_kv_ms']:.2f}, first_token_gen_ms={timings['first_token_gen_ms']:.2f}, total_gen_ms={timings['total_gen_ms']:.2f}")
    if timings.get('ftl_from_fragments_start_ms', -1) > 0:
        logger.info(f"  FTL (从片段处理开始): {timings['ftl_from_fragments_start_ms']:.2f} ms")
    
    # 如果要继续对后续片段生成，需要手动调用 generate_next_token
    # 例如，继续处理 text_fragments_stream[1:]
    # current_llm_response_text = initial_prompt_text + text_fragments_stream[0] + generated_output
    # for next_frag in text_fragments_stream[1:]:
    #     current_llm_response_text += next_frag
    #     next_tok, _ = llm_streamer.generate_next_token(new_text_fragment=next_frag, max_new_tokens=1)
    #     if next_tok: current_llm_response_text += next_tok
    # logger.info(f"  手动继续后的文本 (示例): '{current_llm_response_text}'")
else:
    logger.warning("LLM Streamer 未初始化，跳过实验 1.1。")


## 5. 实验 2: 不使用 KV 缓存进行推理 (基准对比)

步骤：
1. 每次都将当前累积的完整输入文本 (`initial_prompt_text` + 已接收的 `text_fragments`) 提供给 LLM。
2. 生成下一个 token（或少量 tokens）。
3. 记录首个 token 生成延迟和每次生成的耗时。

In [ ]:
if llm_streamer:
    logger.info("\n--- 实验 2: 不使用 KV 缓存进行推理 (generate_without_cache_for_comparison) ---")
    llm_streamer.reset_state() # 确保没有KV缓存
    
    # 调用封装好的对比方法
    # 这个方法内部会模拟逐步构建输入并调用 model.generate()
    # 它应该能返回类似实验1的指标，但对应的是无缓存的情况
    logger.info(f"测试 generate_without_cache_for_comparison: initial_prompt='{initial_prompt_text}', fragments={text_fragments_stream}")
    
    # 这个函数期望的是完整的初始提示，以及一个后续片段的列表。
    # 它会迭代地将初始提示和越来越多的片段组合起来，然后对每个组合调用 model.generate()
    # 并测量首token延迟等指标。
    results_no_cache = llm_streamer.generate_without_cache_for_comparison(
        initial_prompt_text=initial_prompt_text,
        subsequent_text_fragments=text_fragments_stream,
        max_new_tokens_per_step=1 # 每次只生成一个新token
    )
    
    logger.info(f"  不使用缓存 - 最终生成文本: '{results_no_cache['full_generated_text']}'")
    if results_no_cache['first_token_latency_ms'] > 0:
        logger.info(f"  不使用缓存 - 首个 Token 延迟 (从首个片段处理开始): {results_no_cache['first_token_latency_ms']:.2f} ms")
    if results_no_cache['token_generation_latencies_ms']:
        logger.info(f"  不使用缓存 - 平均每 Token 生成耗时: {np.mean(results_no_cache['token_generation_latencies_ms']):.2f} ms")
    logger.info(f"  不使用缓存 - 总耗时: {results_no_cache['total_processing_time_ms']:.2f} ms")

    # 对比用：一次性处理完整输入 (最简单的不使用缓存情况)
    logger.info("\n--- 对比: 一次性处理完整输入 (无缓存) ---")
    llm_streamer.reset_state()
    start_full_gen_time = time.time()
    # StreamLLMInference 的 generate 方法就是直接调用 model.generate
    # 我们需要用它来生成一定数量的token，例如 text_fragments_stream 的片段数量 + 一点点
    num_target_tokens = len(text_fragments_stream) + 5 # 目标是生成大约这么多新token
    
    # 确保我们从 tokenizer 获取输入ID，因为 generate_without_cache_for_comparison 内部也这样做
    inputs = llm_streamer.tokenizer(full_input_text_for_comparison, return_tensors="pt", padding=True).to(llm_streamer.device)
    input_ids_len = inputs.input_ids.shape[1]
    
    generated_ids = llm_streamer.model.generate(
        inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_new_tokens=num_target_tokens, 
        pad_token_id=llm_streamer.tokenizer.eos_token_id, # 重要，避免padding警告
        temperature=0.1, 
        top_p=0.9,
        do_sample=True
    )
    end_full_gen_time = time.time()
    full_gen_duration_ms = (end_full_gen_time - start_full_gen_time) * 1000
    
    # 解码时跳过原始输入部分
    response_ids_full_gen = generated_ids[0][input_ids_len:]
    response_text_full_gen = llm_streamer.tokenizer.decode(response_ids_full_gen, skip_special_tokens=True)
    
    logger.info(f"  一次性生成结果: '{response_text_full_gen}'")
    logger.info(f"  一次性生成总耗时: {full_gen_duration_ms:.2f} ms")
    # 注意: 这种方式难以精确测量首个token的延迟，除非用更复杂的hook或回调
    # generate_without_cache_for_comparison 方法内部有更细致的计时

else:
    logger.error("LLM Streamer 未初始化，跳过实验 2。")


## 6. 结果分析与讨论

对比实验1和实验2的结果：
- **首个 Token 延迟 (FTL)**: 比较有 KV 缓存（特别是预计算后，从第一个实际输入片段开始计时）和无 KV 缓存（从头处理整个增长的序列）的 FTL。
- **后续 Token 平均延迟**: 比较两种情况下，生成后续 token 的平均耗时。
- **KV 缓存大小**: 观察 KV 缓存随输入增长的变化情况。

预期：
- 使用 KV 缓存（尤其是预计算后）应该能显著降低 FTL。
- 对于后续 token，使用 KV 缓存后，模型不需要重新处理已经看过的文本，因此每个新 token 的生成速度应该更快或至少相当（取决于新片段的处理开销）。
- 不使用 KV 缓存时，随着输入序列增长，每次生成的耗时都会增加，因为模型要处理的上下文越来越长。

In [ ]:
logger.info("LLM KV Cache 实验 Notebook 执行完毕。")
logger.info("请查看日志输出，对比不同实验的延迟数据。")

if llm_streamer:
    # 简单总结一下（实际结果会显示在日志中）
    print("\n--- 结果总结提示 ---")
    if 'first_token_latency_ms' in locals() and first_token_latency_ms > 0:
        print(f"带缓存 - 首 Token 延迟: {first_token_latency_ms:.2f} ms (从首个片段输入开始)")
    if 'results_no_cache' in locals() and results_no_cache['first_token_latency_ms'] > 0:
        print(f"无缓存 - 首 Token 延迟: {results_no_cache['first_token_latency_ms']:.2f} ms (从首个片段输入开始)")
    else:
        print("无缓存FTL数据未收集或未在此处打印，请查看 generate_without_cache_for_comparison 的日志输出。")
    
    if 'token_gen_latencies_ms' in locals() and token_gen_latencies_ms:
        print(f"带缓存 - 平均每 Token 生成耗时 (模型报告): {np.mean(token_gen_latencies_ms):.2f} ms")
    if 'results_no_cache' in locals() and results_no_cache['token_generation_latencies_ms']:
        print(f"无缓存 - 平均每 Token 生成耗时: {np.mean(results_no_cache['token_generation_latencies_ms']):.2f} ms")
else:
    print("LLM Streamer 未初始化，无结果可总结。")

---